In [1]:
import openai

from typing import List, Iterator
import pandas as pd
import numpy as np
import os
import pickle
from ast import literal_eval

# Redis client library for Python
import redis
from redis.commands.search.indexDefinition import (
    IndexDefinition,
    IndexType
)
from redis.commands.search.query import Query
from redis.commands.search.field import (
    TextField,
    VectorField
)

# I've set this to our new embeddings model, this can be changed to the embedding model of your choice
EMBEDDING_MODEL = "text-embedding-3-small"

# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

#### Load Data

In [2]:
transcript_df = pd.read_csv('data/embeddings.csv')

In [3]:
transcript_df.head()

,vector_id,id,video_key,chunk_key,content_vector,title_vector
0,0,0,4b6bwcWK6GE,4b6bwcWK6GE_chunk_0,"[0.007409781217575073, -0.004786759149283171, ...","[-0.027406472712755203, -0.012386651709675789,..."
1,1,1,H-XfCl-HpRM,H-XfCl-HpRM_chunk_0,"[0.0029184420127421618, -0.010121479630470276,...","[-0.02722085453569889, 0.0020761173218488693, ..."
2,2,1,H-XfCl-HpRM,H-XfCl-HpRM_chunk_1,"[0.0036486308090388775, -0.019859155640006065,...","[-0.02722085453569889, 0.0020761173218488693, ..."
3,3,1,H-XfCl-HpRM,H-XfCl-HpRM_chunk_2,"[-0.008065965957939625, -0.04014651104807854, ...","[-0.02722085453569889, 0.0020761173218488693, ..."
4,4,1,H-XfCl-HpRM,H-XfCl-HpRM_chunk_3,"[0.02524896338582039, 0.006082533393055201, -0...","[-0.02722085453569889, 0.0020761173218488693, ..."


In [4]:
# Load title dictionary from pickle file
with open('data/title_dict.pkl', 'rb') as f:
    title_dict = pickle.load(f)

title_df = pd.DataFrame(title_dict.items(), columns=['video_key', 'title'])
title_df.head()

,video_key,title
0,4b6bwcWK6GE,Welcome to the Huberman Lab Podcast
1,H-XfCl-HpRM,How Your Brain Works & Changes
2,nm1TxQj9IsQ,Master Your Sleep & Be More Alert When Awake
3,nwSkFq4tyC0,"Using Science to Optimize Sleep, Learning & Me..."
4,NAATB55oxeQ,"How to Defeat Jet Lag, Shift Work & Sleeplessness"


In [5]:
# Load title dictionary from pickle file
with open('data/chunk_dict.pkl', 'rb') as f:
    chunk_dict = pickle.load(f)

chunk_df = pd.DataFrame(chunk_dict.items(), columns=['chunk_key', 'text'])
chunk_df.head()

,chunk_key,text
0,4b6bwcWK6GE_chunk_0,- Welcome to the Huberman Lab Podcast\nwhere w...
1,H-XfCl-HpRM_chunk_0,- Welcome to the Huberman Lab Podcast\nwhere w...
2,H-XfCl-HpRM_chunk_1,which is more a measure\nof your biological ag...
3,H-XfCl-HpRM_chunk_2,If I play the keys on a\npiano in a particular...
4,H-XfCl-HpRM_chunk_3,We come into the world and\nour brain has a ki...


In [6]:
# Join transcript_df with title_df on video_id to add titles
transcript_df = transcript_df.merge(title_df, on='video_key', how='left')
transcript_df = transcript_df.merge(chunk_df, on='chunk_key', how='left')
transcript_df.head()

,vector_id,id,video_key,chunk_key,content_vector,title_vector,title,text
0,0,0,4b6bwcWK6GE,4b6bwcWK6GE_chunk_0,"[0.007409781217575073, -0.004786759149283171, ...","[-0.027406472712755203, -0.012386651709675789,...",Welcome to the Huberman Lab Podcast,- Welcome to the Huberman Lab Podcast\nwhere w...
1,1,1,H-XfCl-HpRM,H-XfCl-HpRM_chunk_0,"[0.0029184420127421618, -0.010121479630470276,...","[-0.02722085453569889, 0.0020761173218488693, ...",How Your Brain Works & Changes,- Welcome to the Huberman Lab Podcast\nwhere w...
2,2,1,H-XfCl-HpRM,H-XfCl-HpRM_chunk_1,"[0.0036486308090388775, -0.019859155640006065,...","[-0.02722085453569889, 0.0020761173218488693, ...",How Your Brain Works & Changes,which is more a measure\nof your biological ag...
3,3,1,H-XfCl-HpRM,H-XfCl-HpRM_chunk_2,"[-0.008065965957939625, -0.04014651104807854, ...","[-0.02722085453569889, 0.0020761173218488693, ...",How Your Brain Works & Changes,If I play the keys on a\npiano in a particular...
4,4,1,H-XfCl-HpRM,H-XfCl-HpRM_chunk_3,"[0.02524896338582039, 0.006082533393055201, -0...","[-0.02722085453569889, 0.0020761173218488693, ...",How Your Brain Works & Changes,We come into the world and\nour brain has a ki...


In [7]:
# Read vectors from strings back into a list
transcript_df['title_vector'] = transcript_df.title_vector.apply(literal_eval)
transcript_df['content_vector'] = transcript_df.content_vector.apply(literal_eval)

# Set vector_id to be a string
transcript_df['video_key'] = transcript_df['video_key'].apply(str)

In [8]:
transcript_df

,vector_id,id,video_key,chunk_key,content_vector,title_vector,title,text
0,0,0,4b6bwcWK6GE,4b6bwcWK6GE_chunk_0,"[0.007409781217575073, -0.004786759149283171, ...","[-0.027406472712755203, -0.012386651709675789,...",Welcome to the Huberman Lab Podcast,- Welcome to the Huberman Lab Podcast\nwhere w...
1,1,1,H-XfCl-HpRM,H-XfCl-HpRM_chunk_0,"[0.0029184420127421618, -0.010121479630470276,...","[-0.02722085453569889, 0.0020761173218488693, ...",How Your Brain Works & Changes,- Welcome to the Huberman Lab Podcast\nwhere w...
2,2,1,H-XfCl-HpRM,H-XfCl-HpRM_chunk_1,"[0.0036486308090388775, -0.019859155640006065,...","[-0.02722085453569889, 0.0020761173218488693, ...",How Your Brain Works & Changes,which is more a measure\nof your biological ag...
3,3,1,H-XfCl-HpRM,H-XfCl-HpRM_chunk_2,"[-0.008065965957939625, -0.04014651104807854, ...","[-0.02722085453569889, 0.0020761173218488693, ...",How Your Brain Works & Changes,If I play the keys on a\npiano in a particular...
4,4,1,H-XfCl-HpRM,H-XfCl-HpRM_chunk_3,"[0.02524896338582039, 0.006082533393055201, -0...","[-0.02722085453569889, 0.0020761173218488693, ...",How Your Brain Works & Changes,We come into the world and\nour brain has a ki...
...,...,...,...,...,...,...,...,...
7462,7462,287,9vQZT2igXN4,9vQZT2igXN4_chunk_35,"[0.03465878590941429, -0.006087542045861483, -...","[-0.03055098094046116, 0.002287362702190876, 0...",How Different Diets Impact Your Health | Dr. C...,the cannabis research community. I\ninvited th...
7463,7463,287,9vQZT2igXN4,9vQZT2igXN4_chunk_36,"[0.018512507900595665, -0.003153152298182249, ...","[-0.03055098094046116, 0.002287362702190876, 0...",How Different Diets Impact Your Health | Dr. C...,that they hadn't eaten any fermented\nfood har...
7464,7464,287,9vQZT2igXN4,9vQZT2igXN4_chunk_37,"[0.020002253353595734, -0.014687786810100079, ...","[-0.03055098094046116, 0.002287362702190876, 0...",How Different Diets Impact Your Health | Dr. C...,"reactive protein, not interlucan 6, not\ntrime..."
7465,7465,287,9vQZT2igXN4,9vQZT2igXN4_chunk_38,"[0.034738194197416306, -0.025965649634599686, ...","[-0.03055098094046116, 0.002287362702190876, 0...",How Different Diets Impact Your Health | Dr. C...,of other things but I I see significant\nimpro...


In [9]:
transcript_df.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7467 entries, 0 to 7466
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   vector_id       7467 non-null   int64 
 1   id              7467 non-null   int64 
 2   video_key       7467 non-null   object
 3   chunk_key       7467 non-null   object
 4   content_vector  7467 non-null   object
 5   title_vector    7467 non-null   object
 6   title           7467 non-null   object
 7   text            7467 non-null   object
dtypes: int64(2), object(6)
memory usage: 466.8+ KB


### Redis

#### Setup

In [10]:
REDIS_HOST =  "localhost"
REDIS_PORT = 6379
REDIS_PASSWORD = "" # default for passwordless Redis

# Connect to Redis
redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD
)
redis_client.ping()

True

#### Creating a Search Index

The below cells will show how to specify and create a search index in Redis. We will

1. Set some constants for defining our index like the distance metric and the index name
2. Define the index schema with RediSearch fields
3. Create the index


In [11]:
# Constants
VECTOR_DIM = len(transcript_df['title_vector'][0]) # length of the vectors
VECTOR_NUMBER = len(transcript_df)                 # initial number of vectors
INDEX_NAME = "embeddings-index"                    # name of the search index
PREFIX = "doc"                                     # prefix for the document keys
DISTANCE_METRIC = "COSINE"                         # distance metric for the vectors (ex. COSINE, IP, L2)

In [12]:
# Define RediSearch fields for each of the columns in the dataset
title = TextField(name="title")
text = TextField(name="text")
video_key = TextField(name="video_key")
chunk_key = TextField(name="chunk_key")
# url = TextField(name="url")

title_embedding = VectorField("title_vector",
    "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC,
        "INITIAL_CAP": VECTOR_NUMBER,
    }
)
text_embedding = VectorField("content_vector",
    "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": VECTOR_DIM,
        "DISTANCE_METRIC": DISTANCE_METRIC,
        "INITIAL_CAP": VECTOR_NUMBER,
    }
)
fields = [title, text, video_key, chunk_key, title_embedding, text_embedding]

In [13]:
# Check if index exists
try:
    redis_client.ft(INDEX_NAME).info()
    print("Index already exists")
except:
    # Create RediSearch Index
    redis_client.ft(INDEX_NAME).create_index(
        fields = fields,
        definition = IndexDefinition(prefix=[PREFIX], index_type=IndexType.HASH)
    )

#### Load Documents into Index

In [14]:
def index_documents(client: redis.Redis, prefix: str, documents: pd.DataFrame):
    records = documents.to_dict("records")
    for doc in records:
        key = f"{prefix}:{str(doc['id'])}"

        # create byte vectors for title and content
        title_embedding = np.array(doc["title_vector"], dtype=np.float32).tobytes()
        content_embedding = np.array(doc["content_vector"], dtype=np.float32).tobytes()

        # replace list of floats with byte vectors
        doc["title_vector"] = title_embedding
        doc["content_vector"] = content_embedding

        client.hset(key, mapping = doc)

In [15]:
index_documents(redis_client, PREFIX, transcript_df)
print(f"Loaded {redis_client.info()['db0']['keys']} documents in Redis search index with name: {INDEX_NAME}")

Loaded 261 documents in Redis search index with name: embeddings-index


#### Running Search Queries

In [16]:
def search_redis(
    openai_client: openai.OpenAI,
    redis_client: redis.Redis,
    user_query: str,
    index_name: str = "embeddings-index",
    vector_field: str = "title_vector",
    return_fields: list = ["title", "text", "chunk_key", "vector_score"],
    hybrid_fields = "*",
    k: int = 10,
) -> List[dict]:

    # Creates embedding vector from user query
    embedded_query = openai_client.embeddings.create(input=user_query,
                                            model=EMBEDDING_MODEL,
                                            ).data[0].embedding

    # Prepare the Query
    base_query = f'{hybrid_fields}=>[KNN {k} @{vector_field} $vector AS vector_score]'
    query = (
        Query(base_query)
         .return_fields(*return_fields)
         .sort_by("vector_score")
         .paging(0, k)
         .dialect(2)
    )
    params_dict = {"vector": np.array(embedded_query).astype(dtype=np.float32).tobytes()}

    # perform vector search
    results = redis_client.ft(index_name).search(query, params_dict)
    for i, article in enumerate(results.docs):
        score = 1 - float(article.vector_score)
        print(f"{i}. {article.title}\n(Score: {round(score ,3) })")
    return results.docs

In [17]:
# For using OpenAI to generate query embedding
from dotenv import load_dotenv
load_dotenv()
open_api_key = os.getenv("OPENAI_API_KEY")
openai_client = openai.OpenAI(api_key=open_api_key)

In [18]:
results = search_redis(
    openai_client, 
    redis_client, 
    'dopamine in the brain', 
    k=10
)

0. Time Perception & Entrainment by Dopamine, Serotonin & Hormones
(Score: 0.428)
1. Dr. Robert Malenka: How Your Brainâs Reward Circuits Drive Your Choices
(Score: 0.403)
2. Leverage Dopamine to Overcome Procrastination & Optimize Effort | Huberman Lab Podcast
(Score: 0.377)
3. How Relationships Shape Your Brain | Dr. Allan Schore
(Score: 0.346)
4. Control Pain & Heal Faster With Your Brain | Huberman Lab Essentials
(Score: 0.342)
5. How Your Brain Works & Changes
(Score: 0.334)
6. Using Play to Rewire & Improve Your Brain
(Score: 0.332)
7. Essentials: How Your Brain Works & Changes
(Score: 0.332)
8. Dr. Mark D'Esposito: How to Optimize Cognitive Function & Brain Health
(Score: 0.329)
9. Optimize & Control Your Brain Chemistry to Improve Health & Performance | Huberman Lab Podcast #80
(Score: 0.314)


In [19]:
results[0].text

"It's simply Huberman Lab on YouTube.\nAnd there you can also\nleave us suggestions\nfor future guests and topics\nand questions about the podcast episodes\nin the comments section on YouTube.\nIn addition, please\nsubscribe to our podcast\non Apple and/or Spotify, and on Apple,\nyou have the opportunity to leave us\nup to a five star review.\nYou can also follow us on Instagram.\nOn Instagram I do short\nneuroscience tutorials\nand tools and protocols.\nI cover recent papers,\nmany of which are not\nincluded on the podcast.\nWe also have a Patreon. It's\npatreon.com/andrewhuberman.\nAnd there you can support the podcast\nat any level that you like.\nNot so much today but on\nmany previous episodes\nof the Huberman Lab Podcast,\nwe discuss supplements,\nand while supplements aren't\nnecessary for everybody,\nmany people derive great\nbenefit from supplements\nfor sleep, for focus, and so forth.\nOne issue with supplements,\nhowever, is that what's listed\non the bottle of various suppl

In [20]:
results = search_redis(
    openai_client, 
    redis_client, 
    'Sleep and Exercise', 
    vector_field='content_vector', 
    k=10
)

0. How to Defeat Jet Lag, Shift Work & Sleeplessness | Huberman Lab Essentials
(Score: 0.494)
1. Dr. Matt Walker: The Biology of Sleep & Your Unique Sleep Needs | Huberman Lab Guest Series
(Score: 0.432)
2. Optimize Your Learning & Creativity With Science-Based Tools | Huberman Lab Essentials
(Score: 0.426)
3. Dr. Gina Poe: Use Sleep to Enhance Learning, Memory & Emotional State | Huberman Lab Podcast
(Score: 0.416)
4. How to Heal From Post-Traumatic Stress Disorder (PTSD) | Dr. Victor CarriÃ³n
(Score: 0.41)
5. Dr. Matt Walker: Improve Sleep to Boost Mood & Emotional Regulation | Huberman Lab Guest Series
(Score: 0.409)
6. Create Your Ideal Future Using Science-Based Protocols | Ari Wallach
(Score: 0.407)
7. Master Your Sleep & Be More Alert When Awake
(Score: 0.406)
8. AMA #14: 2023 Philanthropy, Evening Routine, Light Therapy, Health Metrics & More
(Score: 0.403)
9. Lose Fat With Science-Based Tools | Huberman Lab Essentials
(Score: 0.402)


#### Hybrid Queries with Redis

In [21]:
def create_hybrid_field(field_name: str, value: str) -> str:
    return f'@{field_name}:"{value}"'

In [22]:
# search the content vector for articles about famous battles in Scottish history and only include results with Scottish in the title
results = search_redis(openai_client,
                       redis_client,
                       "Sleep and Exercise",
                       vector_field="title_vector",
                       k=5,
                       hybrid_fields=create_hybrid_field("title", "Huberman")
                       )

0. Using Science to Optimize Sleep, Learning & Metabolism | Huberman Lab Essentials
(Score: 0.521)
1. How to Defeat Jet Lag, Shift Work & Sleeplessness | Huberman Lab Essentials
(Score: 0.488)
2. Dr. Matt Walker: Protocols to Improve Your Sleep | Huberman Lab Guest Series
(Score: 0.47)
3. Sleep Toolkit: Tools for Optimizing Sleep & Sleep-Wake Timing | Huberman Lab Podcast #84
(Score: 0.461)
4. Dr. Matt Walker: How to Structure Your Sleep, Use Naps & Time Caffeine | Huberman Lab Guest Series
(Score: 0.455)


In [23]:
# run a hybrid query for articles about Art in the title vector and only include results with the phrase "Leonardo da Vinci" in the text
results = search_redis(openai_client,
                       redis_client,
                       "Sleep and Exercise",
                       vector_field="title_vector",
                       k=5,
                       hybrid_fields=create_hybrid_field("text", "Huberman")
                       )
# find specific mention of Leonardo da Vinci in the text that our full-text-search query returned
mention = [sentence for sentence in results[0].text.split("\n") if "Huberman" in sentence][0]
mention

0. Master Your Sleep & Be More Alert When Awake
(Score: 0.527)
1. Dr. Samer Hattar: Timing Light, Food, & Exercise for Better Sleep, Energy & Mood
(Score: 0.523)
2. How to Defeat Jet Lag, Shift Work & Sleeplessness
(Score: 0.518)
3. Dr. Matt Walker: Protocols to Improve Your Sleep | Huberman Lab Guest Series
(Score: 0.47)
4. Sleep Toolkit: Tools for Optimizing Sleep & Sleep-Wake Timing | Huberman Lab Podcast #84
(Score: 0.461)


'or check out the Huberman Lab Instagram'

In [24]:
results

[Document {'id': 'doc:2', 'payload': None, 'vector_score': '0.472522139549', 'title': 'Master Your Sleep & Be More Alert When Awake', 'text': "A note about sleepwalkers and\npeople with very vivid dreams,\ntheanine can often make\nyour dreams very vivid,\nsleepwalkers should be\ncareful about taking theanine,\neveryone should be careful\nabout taking anything,\nand don't take anything\nwithout consulting your\nboard-certified M.D.\nor healthcare professional first, okay?\nYour health is your responsibility,\nI am not gonna take responsibility\nfor what you decide to do\nexperimentally in any case,\nbut especially as it relates\nto supplementation and drugs.\nAs a important point, apigenin\nis a fairly potent estrogen inhibitor,\nso women who want to keep\ntheir estrogen levels high,\nor at whatever levels\nthey happen to be at,\nshould probably avoid apigenin altogether,\nand men, take that into\nconsideration as well,\nmen need estrogen also,\nyou don't wanna completely\neliminate you